# I6 — Linear classifiers alongside the winning I5 MLP heads

Open a fresh copy, select T4 GPU, enable the `KAGGLE_API_TOKEN` secret and run all cells. The winning I5 folder `student_outputs_I5_NineHead/5e401a22b20f` is prefilled. Keep its nine head checkpoints, summary, probabilities archive and submission CSV, plus original B336/L336 feature caches under Drive's `student_outputs_I*/cache_336/`.

Six new linear classifiers (768→20 for B336, 1024→20 for L336) use seeds 42/123/2026. Each trains for a **fixed 20 epochs**, AdamW lr/decay .001, batch64, on the saved 3500 training-feature rows from 700 images. Source training mean/std remain fixed. **Validation is diagnostic only: no early stopping, best-epoch selection or export veto based on validation improvement.** No backbone download, image inference, TTA, pseudo-labeling or new feature extraction occurs.

The sole fixed candidate leaves the S branch unchanged, mixes **75% I5 MLP + 25% linear probabilities** within each B/L branch, then retains equal S/B/L backbone weights. Both MLP and linear seed groups use equal seed weights. No alternative blend weights or seed subsets are searched.

All six I5 B/L heads must replay from cached features. I5's S mean is recovered from the saved equal-backbone mean and B/L probabilities, with FP32 bounds and reconstruction checks. The scored I5 submission must reproduce before linear training begins. All nine I5 source checkpoints stay unchanged.

Outputs go to `student_outputs_I6_LinearBlend/<uuid>/`. Keep the full folder. The candidate is exported regardless of validation change; no automatic Kaggle submission occurs. Keep **0.98163** as the reference until the candidate is scored.

In [ ]:
%pip -q install scikit-learn pandas pillow kagglehub

In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
STORAGE_ROOT=Path('/content/drive/MyDrive/Car-Classification-Benchmark')
I5_RUN_DIR=STORAGE_ROOT/'student_outputs_I5_NineHead/5e401a22b20f'
for name in ('summary.json','probabilities.npz','I5_S_B336_L336_three_seed_submission.csv'):
    assert (I5_RUN_DIR/name).is_file(), f'Missing saved I5 file: {I5_RUN_DIR/name}'
DOWNLOAD_FROM_KAGGLE=True
DATA_ROOT=Path('/content/car_data')
if DOWNLOAD_FROM_KAGGLE:
    import kagglehub
    if not os.environ.get('KAGGLE_API_TOKEN'):
        from google.colab import userdata
        os.environ['KAGGLE_API_TOKEN']=userdata.get('KAGGLE_API_TOKEN')
    DATA_ROOT=Path(kagglehub.competition_download('idnn-swe-3052-assignment-3-flowers-classification'))
for split in ('train','val','test'):
    assert (DATA_ROOT/split).is_dir(), f'Missing dataset split: {DATA_ROOT/split}'
print('Data:',DATA_ROOT,'I5 source:',I5_RUN_DIR)


In [ ]:
"""I6: fixed linear/MLP mixtures on cached B336/L336 features; preserve I5."""
from pathlib import Path
import gc, hashlib, json, random, shutil, uuid
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets

LINEAR_RECIPE={'epochs':20,'batch_size':64,'learning_rate':.001,'weight_decay':.001,
               'optimizer':'AdamW','selection':'final epoch; validation diagnostic only'}
LINEAR_WEIGHT=.25


def validate_probability(p,n):
    assert p.shape==(n,20) and p.dtype==torch.float32 and torch.isfinite(p).all()
    assert (p>=0).all() and (p<=1).all()
    assert torch.allclose(p.sum(1),torch.ones(n),atol=1e-5)


def recover_small(current,base,large):
    for p in (current,base,large):validate_probability(p,len(current))
    small=3*current-base-large
    assert torch.isfinite(small).all() and (small>=-2e-6).all() and (small<=1+2e-6).all()
    assert torch.allclose(small.sum(1),torch.ones(len(small)),atol=2e-5)
    # Subtraction can introduce tiny FP32 negatives for near-zero probabilities.
    small=small.clamp(0,1);small=small/small.sum(1,keepdim=True)
    validate_probability(small,len(current))
    assert torch.allclose(equal_blend(small,base,large),current,atol=2e-6,rtol=1e-5)
    return small


def linear_mlp_blend(small,base,large,linear_base,linear_large):
    for p in (small,base,large,linear_base,linear_large):validate_probability(p,len(small))
    return equal_blend(small,(1-LINEAR_WEIGHT)*base+LINEAR_WEIGHT*linear_base,
        (1-LINEAR_WEIGHT)*large+LINEAR_WEIGHT*linear_large)


@torch.inference_mode()
def linear_probabilities(head,source,features,device,keys=('X_val','X_test')):
    head.eval();result=[]
    for key in keys:
        x=(features[key]-source['feature_mean'])/source['feature_std']
        p=head(x.to(device)).softmax(1).cpu();validate_probability(p,len(x));result.append(p)
    return result


def train_linear(seed,source,features,output,device):
    assert seed in SEEDS
    dim=len(source['feature_mean']);set_seed(seed);head=nn.Linear(dim,20).to(device)
    loader=DataLoader(TensorDataset((features['X_train']-source['feature_mean'])/source['feature_std'],features['y_train']),
        batch_size=64,shuffle=True,num_workers=0,generator=torch.Generator().manual_seed(seed))
    optimizer=torch.optim.AdamW(head.parameters(),lr=LINEAR_RECIPE['learning_rate'],weight_decay=LINEAR_RECIPE['weight_decay'])
    history=[]
    for epoch in range(1,LINEAR_RECIPE['epochs']+1):
        head.train();total=0.
        for x,y in loader:
            optimizer.zero_grad(set_to_none=True)
            loss=nn.functional.cross_entropy(head(x.to(device)),y.to(device));assert torch.isfinite(loss)
            loss.backward();optimizer.step();total+=loss.item()*len(y)
        pv=linear_probabilities(head,source,features,device,keys=('X_val',))[0]
        score=metrics(features['y_val'].numpy(),pv)
        history.append({'epoch':epoch,'train_loss':total/len(features['X_train']),**score})
        print('I6_LINEAR',source['backbone'],'SEED',seed,'EPOCH',epoch,score,flush=True)
    pv,pt=linear_probabilities(head,source,features,device)
    checkpoint={'head_state_dict':{k:v.detach().cpu().clone() for k,v in head.state_dict().items()},
        'head_config':{'kind':'linear','input_dim':dim,'num_classes':20},'backbone':source['backbone'],
        'hub_ref':HUB_REF,'resolution':336,'seed_head':seed,'seed_aug':123,'selected_epoch':LINEAR_RECIPE['epochs'],
        'class_to_idx':source['class_to_idx'],'feature_mean':source['feature_mean'],'feature_std':source['feature_std'],
        'feature_metadata':source['feature_metadata'],'feature_signature':source['feature_signature'],
        'metrics':score,'training_config':LINEAR_RECIPE}
    name=f"I6_{source['backbone']}_linear_seed{seed}"
    path=Path(output)/(name+'.pt')
    with path.open('xb') as stream:torch.save(checkpoint,stream)
    pd.DataFrame(history).to_csv(Path(output)/(name+'_history.csv'),index=False)
    return pv,pt,path,checkpoint


def load_i5_sources(i5_run,mapping):
    i5_run=Path(i5_run);summary=json.loads((i5_run/'summary.json').read_text())
    assert summary['head_seeds']==SEEDS and np.allclose(summary['effective_head_weights'],[1/9]*9)
    assert summary['TTA'] is False and summary['backbones_frozen'] is True
    expected={Path(p).name:h for p,h in {**summary['source_checkpoint_sha256'],**summary['new_checkpoint_sha256']}.items()}
    assert len(expected)==9
    groups={name:{} for name in ('dinov2_vits14','dinov2_vitb14','dinov2_vitl14')};paths=[]
    for name,digest in expected.items():
        path=i5_run/name;assert sha(path)==digest,'I5 source checkpoint changed: '+name
        c=torch.load(path,map_location='cpu',weights_only=True)
        model=c['backbone'];dim={'dinov2_vits14':384,'dinov2_vitb14':768,'dinov2_vitl14':1024}[model]
        assert c['class_to_idx']==mapping and c['seed_aug']==123 and c['seed_head'] in SEEDS
        assert c['seed_head'] not in groups[model]
        for key in ('feature_mean','feature_std'):assert c[key].shape==(dim,) and torch.isfinite(c[key]).all()
        assert (c['feature_std']>0).all()
        build_head(dim).load_state_dict(c['head_state_dict'],strict=True)
        if dim!=384:assert c['resolution']==336 and c['hub_ref']==HUB_REF
        groups[model][c['seed_head']]=c;paths.append(path)
    for heads in groups.values():assert set(heads)==set(SEEDS)
    return summary,groups,paths,{str(i5_run/name):digest for name,digest in expected.items()}


def run_i6(data_root,i5_run,storage_root):
    set_seed(2026);device='cuda' if torch.cuda.is_available() else 'cpu'
    mapping,_,target,_,ids=load_data(data_root);i5_run=Path(i5_run)
    reference,groups,source_paths,original=load_i5_sources(i5_run,mapping)
    artifacts={str(i5_run/name):sha(i5_run/name) for name in ('summary.json','probabilities.npz','I5_S_B336_L336_three_seed_submission.csv')}
    base=groups['dinov2_vitb14'][2026];large=groups['dinov2_vitl14'][2026]
    validate_source_data(base,data_root,mapping,ids)
    assert large['feature_metadata']['manifest']==base['feature_metadata']['manifest']
    with np.load(i5_run/'probabilities.npz',allow_pickle=False) as saved:
        assert np.array_equal(saved['targets'],target)
        current_v=torch.from_numpy(saved['validation_candidate'].copy())
        current_t=torch.from_numpy(saved['test_candidate'].copy())
        archived={branch:(torch.from_numpy(saved[branch+'_validation_heads'].copy()),torch.from_numpy(saved[branch+'_test_heads'].copy())) for branch in ('B336','L336')}
    validate_probability(current_v,len(target));validate_probability(current_t,len(ids))
    reproduce(metrics(target,current_v),reference['candidate_validation'],'Winning I5 validation')
    control_path=i5_run/'I5_S_B336_L336_three_seed_submission.csv'
    control=pd.read_csv(control_path,dtype={'image_id':str})
    assert control.image_id.tolist()==ids and np.array_equal(control.label.to_numpy(),current_t.argmax(1).numpy()),'I5 CSV/archived probabilities differ'
    root=Path(storage_root)/'student_outputs_I6_LinearBlend';output=root/uuid.uuid4().hex[:12];output.mkdir(parents=True,exist_ok=False)
    print('OUTPUT',output,'DEVICE',device,flush=True)
    print('Fixed candidate: S unchanged; B/L each 75% I5 MLP + 25% linear; equal S/B/L branches.',flush=True)
    branches={};linear={};new_paths=[];new_metrics=[];cache_sources={}
    for branch,model,source in [('B336','dinov2_vitb14',base),('L336','dinov2_vitl14',large)]:
        features,cache=full_feature_cache(source,storage_root,data_root,target,ids)
        val_parts=[];test_parts=[]
        for index,seed in enumerate(SEEDS):
            checkpoint=groups[model][seed]
            assert checkpoint['feature_metadata']==source['feature_metadata'] and checkpoint['feature_signature']==source['feature_signature']
            assert torch.equal(checkpoint['feature_mean'],source['feature_mean']) and torch.equal(checkpoint['feature_std'],source['feature_std'])
            pv,pt=head_probabilities(checkpoint,features,device)
            assert archived[branch][0].shape==(3,len(target),20) and archived[branch][1].shape==(3,len(ids),20)
            assert torch.allclose(pv,archived[branch][0][index],atol=2e-5,rtol=2e-4)
            assert torch.allclose(pt,archived[branch][1][index],atol=2e-5,rtol=2e-4)
            reproduce(metrics(target,pv),checkpoint['metrics'],branch+' seed'+str(seed))
            val_parts.append(archived[branch][0][index]);test_parts.append(archived[branch][1][index])
        branches[branch]=(torch.stack(val_parts).mean(0),torch.stack(test_parts).mean(0))
        # All six source heads must replay before any linear training starts.
        cache_sources[branch]={'path':str(cache),'signature':source['feature_signature']}
        del features;gc.collect()
    small_v=recover_small(current_v,branches['B336'][0],branches['L336'][0])
    small_t=recover_small(current_t,branches['B336'][1],branches['L336'][1])
    assert np.array_equal(equal_blend(small_t,branches['B336'][1],branches['L336'][1]).argmax(1).numpy(),control.label.to_numpy())
    print('I5 source replay passed. Beginning fixed linear-head training.',flush=True)
    for branch,source in [('B336',base),('L336',large)]:
        features,_=full_feature_cache(source,storage_root,data_root,target,ids)
        vals=[];tests=[]
        for seed in SEEDS:
            pv,pt,path,checkpoint=train_linear(seed,source,features,output,device)
            vals.append(pv);tests.append(pt);new_paths.append(path)
            new_metrics.append({'branch':branch,'seed':seed,**checkpoint['metrics']})
        linear[branch]=(torch.stack(vals).mean(0),torch.stack(tests).mean(0))
        del features;gc.collect()
    candidate_v=linear_mlp_blend(small_v,branches['B336'][0],branches['L336'][0],linear['B336'][0],linear['L336'][0])
    candidate_t=linear_mlp_blend(small_t,branches['B336'][1],branches['L336'][1],linear['B336'][1],linear['L336'][1])
    for path in source_paths:
        assert sha(path)==original[str(path)],'I5 source checkpoint modified'
        with (output/path.name).open('xb') as stream,path.open('rb') as source_stream:shutil.copyfileobj(source_stream,stream)
        assert sha(output/path.name)==original[str(path)]
    for name,digest in artifacts.items():assert sha(name)==digest,'I5 reference artifact changed'
    export(output/'CurrentBest_098163_control.csv',ids,current_t)
    export(output/'I6_S_B336_L336_MLP_linear_blend_submission.csv',ids,candidate_t)
    changed=current_t.argmax(1)!=candidate_t.argmax(1)
    pd.DataFrame({'image_id':ids,'original_label':current_t.argmax(1).numpy(),'candidate_label':candidate_t.argmax(1).numpy(),'changed':changed.numpy()}).to_csv(output/'test_prediction_changes.csv',index=False)
    summary={'experiment':'I6 fixed MLP/linear head blend','linear_weight_within_B_L':LINEAR_WEIGHT,
        'branch_weights':[1/3]*3,'head_seeds':SEEDS,'linear_recipe':LINEAR_RECIPE,
        'per_head_weights':{'S_MLP':1/9,'B_L_MLP':1/12,'B_L_linear':1/36},
        'original_validation':metrics(target,current_v),'candidate_validation':metrics(target,candidate_v),
        'linear_heads_validation':new_metrics,'test_changed':int(changed.sum()),'cache_sources':cache_sources,
        'source_checkpoint_sha256':original,'source_artifact_sha256':artifacts,'new_checkpoint_sha256':{str(p):sha(p) for p in new_paths},
        'reference_submission_sha256':sha(control_path),'reference_probabilities_sha256':sha(i5_run/'probabilities.npz'),
        'historical_best_kaggle':.98163,'source_predictions_reproduced':True,'validation_selection':False,
        'S_branch':'recovered from saved I5 mean and replayed B/L probabilities; FP32 tolerance checked',
        'TTA':False,'backbones_frozen':True,'test_labels_used':False,'submitted':False}
    np.savez_compressed(output/'probabilities.npz',validation_original=current_v.numpy(),validation_candidate=candidate_v.numpy(),
        test_original=current_t.numpy(),test_candidate=candidate_t.numpy(),targets=target,
        S_validation=small_v.numpy(),S_test=small_t.numpy(),
        B336_MLP_validation=branches['B336'][0].numpy(),B336_MLP_test=branches['B336'][1].numpy(),
        L336_MLP_validation=branches['L336'][0].numpy(),L336_MLP_test=branches['L336'][1].numpy(),
        B336_linear_validation=linear['B336'][0].numpy(),B336_linear_test=linear['B336'][1].numpy(),
        L336_linear_validation=linear['L336'][0].numpy(),L336_linear_test=linear['L336'][1].numpy())
    (output/'summary.json').write_text(json.dumps(summary,indent=2))
    print('COMPLETE',output,json.dumps(summary),flush=True)
    return output,summary

HUB_REF='7764ea0f912e53c92e82eb78a2a1631e92725fc8'
SEEDS=[42,123,2026]
def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as stream:
        for chunk in iter(lambda:stream.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()

def build_head(dim):
    return nn.Sequential(nn.Linear(dim,128),nn.BatchNorm1d(128),nn.ReLU(),nn.Dropout(.3),nn.Linear(128,20))

def load_data(root):
    root=Path(root)
    train,val=[datasets.ImageFolder(root/name) for name in ('train','val')]
    assert len(train)==700 and len(val)==200 and len(train.classes)==20
    assert train.class_to_idx==val.class_to_idx
    assert all(name.startswith(f'{i:03d}_') for i,name in enumerate(train.classes))
    sample=pd.read_csv(root/'sample_submission.csv',dtype={'image_id':str})
    assert {'image_id','label'}<=set(sample.columns) and sample.image_id.is_unique and sample.image_id.notna().all()
    files=sorted(p for p in (root/'test').iterdir() if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'})
    by_id={p.stem:p for p in files};ids=sample.image_id.tolist()
    assert len(by_id)==len(files) and set(by_id)==set(ids)
    return train.class_to_idx,[Path(p) for p,_ in val.samples],np.asarray(val.targets),[by_id[i] for i in ids],ids

def metrics(target,probabilities):
    from sklearn.metrics import accuracy_score,f1_score
    pred=probabilities.argmax(1).numpy()
    return {'accuracy':float(accuracy_score(target,pred)),
        'macro_f1':float(f1_score(target,pred,labels=list(range(20)),average='macro',zero_division=0)),
        'correct':int((pred==target).sum()),'count':len(target)}

def reproduce(measured,expected,name):
    assert measured['correct']==expected['correct'] and abs(measured['accuracy']-expected['accuracy'])<1e-12, (
        f'{name} starting accuracy did not reproduce; diagnose checkpoint/source/preprocessing before exporting')

def export(path,ids,probabilities):
    pred=probabilities.argmax(1).numpy().astype(int)
    assert len(pred)==len(ids)==len(set(ids)) and ((pred>=0)&(pred<20)).all()
    pd.DataFrame({'image_id':list(ids),'label':pred}).to_csv(path,index=False,mode='x')

def set_seed(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True
    torch.use_deterministic_algorithms(True)

def validate_source_data(base,data_root,mapping,ids):
    meta=base['feature_metadata']
    assert meta.get('backbone',meta.get('model'))=='dinov2_vitb14'
    assert base['hub_ref']==HUB_REF and meta['class_to_idx']==mapping and meta['test_ids']==ids
    assert meta['feature_dim']==768 and meta['seed_aug']==123 and meta['n_aug']==4 and meta['include_original'] is True
    root=Path(data_root).resolve()
    for split in meta['manifest']:
        for entry in (split['files'] if isinstance(split,dict) else split):
            path=(root/entry['path']).resolve()
            assert path.is_relative_to(root) and path.is_file() and sha(path)==entry['sha256'],'Data differs from original I0 run'

def full_feature_cache(source,storage_root,data_root,target,ids):
    meta=source['feature_metadata'];dim=len(source['feature_mean'])
    signature=hashlib.sha256(json.dumps(meta,sort_keys=True).encode()).hexdigest()
    assert signature==source['feature_signature']
    paths=list(Path(storage_root).glob('student_outputs_I*/cache_336/*_'+signature+'.pt'))
    assert len(paths)==1,'Restore the original feature cache: '+signature
    p=torch.load(paths[0],map_location='cpu',weights_only=True)
    assert p['metadata']==meta and meta['test_ids']==ids and meta['image_size']==336
    assert meta['feature_dim']==dim and meta['seed_aug']==123 and meta['n_aug']==4
    assert meta['include_original'] is True and meta['layout']=='view-major'
    training=datasets.ImageFolder(Path(data_root)/'train')
    assert training.class_to_idx==source['class_to_idx'] and len(training)==700
    for key,n in [('X_train',3500),('X_val',len(target)),('X_test',len(ids))]:
        assert p[key].shape==(n,dim) and p[key].dtype==torch.float32 and torch.isfinite(p[key]).all()
    assert torch.equal(p['y_train'],torch.tensor(training.targets).repeat(5))
    assert np.array_equal(p['y_val'].numpy(),target)
    assert torch.allclose(p['X_train'].mean(0),source['feature_mean'],rtol=1e-5,atol=1e-6)
    assert torch.allclose(p['X_train'].std(0,unbiased=False).clamp_min(1e-6),source['feature_std'],rtol=1e-5,atol=1e-6)
    return p,paths[0]

@torch.inference_mode()
def head_probabilities(source,features,device):
    dim=len(source['feature_mean']);head=build_head(dim).to(device)
    head.load_state_dict(source['head_state_dict']);head.eval()
    result=[]
    for key in ('X_val','X_test'):
        x=(features[key]-source['feature_mean'])/source['feature_std']
        p=head(x.to(device)).softmax(1).cpu()
        assert p.shape==(len(x),20) and torch.isfinite(p).all()
        result.append(p)
    return result

def equal_blend(small,base,large):
    assert small.shape==base.shape==large.shape and small.ndim==2 and small.shape[1]==20
    for p in (small,base,large):
        assert torch.isfinite(p).all() and (p>=0).all() and (p<=1).all()
        assert torch.allclose(p.sum(1),torch.ones(len(p)),atol=1e-5)
    return (small+base+large)/3



In [ ]:
output,summary=run_i6(DATA_ROOT,I5_RUN_DIR,STORAGE_ROOT)
from IPython.display import display
display(pd.DataFrame([{'model':name,**summary[key]} for name,key in [('I5 original','original_validation'),('I6 linear blend','candidate_validation')]]))
print('Changed test predictions:',summary['test_changed'])
print('New candidate (not submitted):',output/'I6_S_B336_L336_MLP_linear_blend_submission.csv')
print('Original control:',output/'CurrentBest_098163_control.csv')
print('Preserve the complete output folder:',output)